In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

All libraries imported successfully
Pandas version: 2.3.3
Numpy version: 2.4.4


In [4]:
df_peek = pd.read_csv('../data/household_power_consumption.txt',
 sep=';', 
 nrows=100,
 na_values = '?',
 low_memory = False)

print("Shape:", df_peek.shape)
print("\nColumn names:")
print(df_peek.columns.tolist())
print("\nFirst 5 rows:")
df_peek.head()

Shape: (100, 9)

Column names:
['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

First 5 rows:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [5]:
# understand data types
print("Data types of each column:")
print(df_peek.dtypes)
print("\nMemory usage by these 100 rows:")
print(f"{df_peek.memory_usage(deep=True).sum() / 1024:.2f} KB")

Data types of each column:
Date                      object
Time                      object
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

Memory usage by these 100 rows:
16.93 KB


In [11]:
# understand what each column represents

column_explanations = {
    'Date': 'Date of measurement (DD/MM/YYYY)',
    'Time': 'Time of measurement (HH:MM:SS) - 1 reading per minute',
    'Global_active_power': 'MAIN SIGNAL - total power consumed (kilowatts). Normal: 0.2-6kW',
    'Global_reactive_power': 'Power stored/released by motors (kilowatts). High = possible fault',
    'Voltage': 'Household voltage (volts). Normal Europe: 220-240V. Outside = anomaly',
    'Global_intensity': 'Current intensity (amperes). Very high = overload risk',
    'Sub_metering_1': 'Kitchen appliances - dishwasher, microwave, oven (watt-hours)',
    'Sub_metering_2': 'Laundry room - washing machine, dryer, fridge (watt-hours)',
    'Sub_metering_3': 'Water heater + air conditioner (watt-hours)'
}

print("="*65)
print("WHAT EACH COLUMN MEANS")
print("="*65)
for col, explanation in column_explanations.items():
    print(f"\n{col}:")
    print(f"   {explanation}")

print("\n" + "="*65)
print("KEY INSIGHT FOR ANOMALY DETECTION:")
print("="*65)
print("""
If Global_active_power is HIGH but all Sub_metering columns
are LOW - power is being consumed by an UNKNOWN source. This is a classic anomaly - could be energy theft, a faulty
meter, or an unregistered appliance.
Our Isolation Forest will learn to catch exactly this pattern.
""")

WHAT EACH COLUMN MEANS

Date:
   Date of measurement (DD/MM/YYYY)

Time:
   Time of measurement (HH:MM:SS) - 1 reading per minute

Global_active_power:
   MAIN SIGNAL - total power consumed (kilowatts). Normal: 0.2-6kW

Global_reactive_power:
   Power stored/released by motors (kilowatts). High = possible fault

Voltage:
   Household voltage (volts). Normal Europe: 220-240V. Outside = anomaly

Global_intensity:
   Current intensity (amperes). Very high = overload risk

Sub_metering_1:
   Kitchen appliances - dishwasher, microwave, oven (watt-hours)

Sub_metering_2:
   Laundry room - washing machine, dryer, fridge (watt-hours)

Sub_metering_3:
   Water heater + air conditioner (watt-hours)

KEY INSIGHT FOR ANOMALY DETECTION:

If Global_active_power is HIGH but all Sub_metering columns
are LOW - power is being consumed by an UNKNOWN source. This is a classic anomaly - could be energy theft, a faulty
meter, or an unregistered appliance.
Our Isolation Forest will learn to catch exactly thi

In [13]:
# check missing values in our 100-row sample

print("\nMissing values in each column:")
print(df_peek.isna().sum())

total_missing = df_peek.isna().sum().sum()
total_cells = df_peek.shape[0] * df_peek.shape[1]

print(f"\nTotal missing cells: {total_missing}")
print(f"Total cells: {total_cells}")
print(f"Missing percentage: {total_missing/total_cells*100:.2f}%")

if total_missing == 0:
    print("\nNo missing values in first 100 rows.")
else:
    print(f"\n{total_missing} missing values found even in 100 rows.")


Missing values in each column:
Date                     0
Time                     0
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

Total missing cells: 0
Total cells: 900
Missing percentage: 0.00%

No missing values in first 100 rows.


In [15]:
# find extreme readings in our 100 rows

print("="*55)
print("HIGHEST power consumption in first 100 rows:")
print("="*55)
print(df_peek.loc[df_peek['Global_active_power'].idxmax()])

print("\n" + "="*55)
print("LOWEST power consumption in first 100 rows:")
print("="*55)
print(df_peek.loc[df_peek['Global_active_power'].idxmin()])

print("\n" + "="*55)
print("VOLTAGE analysis:")
print("="*55)
print(f"Min voltage: {df_peek['Voltage'].min():.2f} V")
print(f"Max voltage: {df_peek['Voltage'].max():.2f} V")
print(f"Average voltage: {df_peek['Voltage'].mean():.2f} V")
print(f"\nNormal range: 220-240V")
print(f"All readings in normal range: {((df_peek['Voltage'] >= 220) & (df_peek['Voltage'] <= 240)).all()}")

print("\n" + "="*55)
print("SUB_METERING breakdown at peak consumption:")
print("="*55)
peak_row = df_peek.loc[df_peek['Global_active_power'].idxmax()]
print(f"Time of peak: {peak_row['Date']} {peak_row['Time']}")
print(f"Total power: {peak_row['Global_active_power']} kW")
print(f"Kitchen (Sub1): {peak_row['Sub_metering_1']} Wh")
print(f"Laundry (Sub2): {peak_row['Sub_metering_2']} Wh")
print(f"Heater/AC (Sub3): {peak_row['Sub_metering_3']} Wh")
print(f"\nUnaccounted power = Total - (Sub1+Sub2+Sub3)/1000")
unaccounted = peak_row['Global_active_power'] - (peak_row['Sub_metering_1'] + peak_row['Sub_metering_2'] + peak_row['Sub_metering_3'])/1000
print(f"Unaccounted power: {unaccounted:.3f} kW")
print("(This is power used by unlisted appliances - TVs, computers, lights)")

HIGHEST power consumption in first 100 rows:
Date                     16/12/2006
Time                       17:45:00
Global_active_power           7.706
Global_reactive_power           0.0
Voltage                      230.98
Global_intensity               33.2
Sub_metering_1                  0.0
Sub_metering_2                  0.0
Sub_metering_3                 17.0
Name: 21, dtype: object

LOWEST power consumption in first 100 rows:
Date                     16/12/2006
Time                       18:43:00
Global_active_power           2.188
Global_reactive_power         0.068
Voltage                       235.8
Global_intensity                9.2
Sub_metering_1                  0.0
Sub_metering_2                  1.0
Sub_metering_3                 17.0
Name: 79, dtype: object

VOLTAGE analysis:
Min voltage: 230.98 V
Max voltage: 238.28 V
Average voltage: 234.51 V

Normal range: 220-240V
All readings in normal range: True

SUB_METERING breakdown at peak consumption:
Time of peak: 16/12/2